# Ejercicio Práctico de QB: modelos desde 0

In [1]:
!mamba install pandas numpy scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 0.274 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ brotli-python       1.2.0       py313h8faef1e_3      emscripten-forge-4x
+ ca-certificates     2026.10.23  hbd8a1cb_0           conda-forge
+ certifi             2026.10.23  pyhd8ed1ab_3         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.28.0      pyh5ded981_0         conda-forge
+ pandas              3.1.0       np23py313h124dbc7_0  emscripten-forge-4x
+ pysocks             1.7.1       py313h1804a44_3      emscripten-forge-4x
+ request

In [2]:
import pandas as pd #Permite leer el archivo y trabajar con el como una 
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Models
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Caso: predecir retrasos en pedidos

Una empresa de distribución quiere identificar qué pedidos tienen más riesgo de llegar tarde para que su equipo de operaciones pueda priorizarlos.

Dispones del archivo pedidospractica.csv. Cada fila representa un pedido.
- pedido_id: Identificador del pedido
- fecha_pedido: Fecha y hora de creación
- tienda: Tienda que prepara el pedido
- zona: Zona de entrega
- distancia_km: Distancia prevista del recorrido
- paradas: Número de paradas planificadas
- plazo_h: Plazo de entrega comprometido, en horas
- retraso: Resultado: 1 si llegó tarde y 0 si llegó a tiempo

### Supuesto: la distancia, las paradas planificadas y el plazo se conocen cuando se crea el pedido. retraso se conoce después de la entrega.

Tu encargo: Construye un modelo que, al crear un nuevo pedido, prediga si llegará con retraso. Explica tus decisiones mientras programas.



Como tengo que predecir una variable binaria: si/no, 0/1 tengo que hacer un modelo de clasificación. Empiezo importando los datos y observandolos

In [5]:
df = pd.read_csv("pedidospractica.csv") #df: dataframe, tiene estructura de tabla, ahi metemos los datos

In [31]:
print(df.head())       # Ver primeras 5 filas

   pedido_id        fecha_pedido tienda    zona  distancia_km  paradas  \
0          1 2025-01-01 00:00:00      F  centro         106.0        0   
1          2 2025-01-01 08:00:00      A     sur          59.0        0   
2          3 2025-01-01 16:00:00      G   norte          22.0        1   
3          4 2025-01-02 00:00:00      B  centro         166.0        4   
4          5 2025-01-02 08:00:00      G  centro          61.0        0   

   plazo_h  retraso  hora_pedido  dia_semana  
0       12        0            0           2  
1       12        0            8           2  
2       12        0           16           2  
3       12        1            0           3  
4       24        0            8           3  


In [32]:
print(df.shape) #Numero de filas y columnas

(300, 10)


In [33]:
print(df.info()) #Como todos son valores no nulos no tengo que eliminar los Na, veo que tengo variables numericas y categoricas
print(df.describe()) #Estadisticas de las variables numericas 

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   pedido_id     300 non-null    int64         
 1   fecha_pedido  300 non-null    datetime64[us]
 2   tienda        300 non-null    str           
 3   zona          300 non-null    str           
 4   distancia_km  288 non-null    float64       
 5   paradas       300 non-null    int64         
 6   plazo_h       300 non-null    int64         
 7   retraso       300 non-null    int64         
 8   hora_pedido   300 non-null    int32         
 9   dia_semana    300 non-null    int32         
dtypes: datetime64[us](1), float64(1), int32(2), int64(4), str(2)
memory usage: 18.8 KB
None
        pedido_id         fecha_pedido  distancia_km     paradas     plazo_h  \
count  300.000000                  300    288.000000  300.000000  300.000000   
mean   150.500000  2025-02-19 20:00:00    101.298611 

De aquí es importante ver el count de retraso, como todos los valores de la variable objetivo están completos no tengo porque eliminar ninguno.
También veo que en distancia_km si que faltan algunos valores, esto lo resolveremos en la preparación de variables

Hay varias observaciones con fecha_pedido
- Es una variable que aporta información importante al modelo
- Pero por el formato que tiene si hacemos OneHotEnconder, al ser una variable categorica se estaría tratando cada fecha completa como una cateogría diferente por lo cual no nos serviría
- Tendríamos 2 opciones: o la quitamos del modelo por simplificarlo o extraemos los datos: hora del pedido y dia de la semana

In [27]:
# Convierto el texto a una fecha que pandas pueda interpretar
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])

# Extraigo información de la fecha
df["hora_pedido"] = df["fecha_pedido"].dt.hour
df["dia_semana"] = df["fecha_pedido"].dt.day_of_week  # Lunes=0, domingo=6
print(df[["fecha_pedido", "hora_pedido", "dia_semana"]].head())
#Al poner df ya he creado las variables de hora_pedido y dia_semana en df

         fecha_pedido  hora_pedido  dia_semana
0 2025-01-01 00:00:00            0           2
1 2025-01-01 08:00:00            8           2
2 2025-01-01 16:00:00           16           2
3 2025-01-02 00:00:00            0           3
4 2025-01-02 08:00:00            8           3


In [28]:
target="retraso"
#No hay variables que me den data leakage asi que solo excluyo pedido_id porque no me sirve
excluir=["pedido_id", "fecha_pedido"]

y=df[target]
X=df.drop(columns=[target] + excluir)

In [29]:
print(X) #Para ver como ha quedado X, sin la variable de retraso, no hay data leakage

    tienda    zona  distancia_km  paradas  plazo_h  hora_pedido  dia_semana
0        F  centro         106.0        0       12            0           2
1        A     sur          59.0        0       12            8           2
2        G   norte          22.0        1       12           16           2
3        B  centro         166.0        4       12            0           3
4        G  centro          61.0        0       24            8           3
..     ...     ...           ...      ...      ...          ...         ...
295      G  centro           8.0        0       18            8           2
296      G     sur         183.0        2       24           16           2
297      G     sur          61.0        3       18            0           3
298      D  centro          38.0        0       24            8           3
299      D  centro          95.0        3       18           16           3

[300 rows x 7 columns]


In [43]:
#Hacemos la separacion de los datos en train y test
#Elegimos usar el 20% de los datos para hacer el test y el 80% para el entrenamiento
#Elegimos una semilla reproducible
#Ponemos stratify para que se quede la distribución de clases porque estamos haciendo un modelo de clasificación

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y) 

#Vuelvo a separar de los datos del train, reservo una parte para la validación
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train)

#Reservo un test para la evaluación final
#Utilizo entrenamiento para ajustar los modelos
#Utilizo validación para compararlos y elgir sus hiperparámetros

In [44]:
#Separamos las variables que tenemos entre numericas y categoricas
var_numericas= ["distancia_km", "paradas", "plazo_h", "hora_pedido", "dia_semana"]
var_categoricas=["tienda", "zona"]

In [46]:
#Ahora vamos a preparar las variables: 
#Variables numericas: rellenar los valores faltantes con la mediana y estandarizar las variables
#Variables categoricas: rellenar los valores faltantes con la categoría más frecuente

preprocesador = ColumnTransformer([
    ("num", make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler() #Para Decision Tree y RF quito StandardScaler
    ), var_numericas),

    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore")
    ), var_categoricas)
])


In [47]:
X_train_prep = preprocesador.fit_transform(X_train)
X_valid_prep = preprocesador.transform(X_valid)

## Modelo 1: Regresión logística
- Algoritmo de clasificación que estima la probabilidad de pertenecer a una clase
- Cada coeficiente indica cómo cambia la probabilidad estimada de y cuando aumenta esa variable
- Aplica a una función logística una combinación lineal de las variables de entrada para obtener una probabilidad entre 0 y 1
- Cuando utilizarla: como primera referencia, para interpretar las relaciones entre variables, cuando una relación lineal es suficiente
- Limitación: no incorpora automñaticamente relaciones no lineales ni interacciones
- Escalado: recomendable (es cuando transformas variables para que tengan magnitudes comparables)

In [50]:
modelo1 = LogisticRegression(
    C=1, #Intensidad de regularización (cuánto se penalizan los coeficientes grandes): menor-más regularización, mayor-más libertad para ajustarse al entrenamiento
    class_weight="balanced", #El peso de las clases, le da mas peso a la clase menos frecuente
    max_iter=1000) #máximo de iteraciones para encontrar los coeficientes

modelo1.fit(X_train_prep, y_train)

y_pred1 = modelo1.predict(X_valid_prep)

print("MODELO 1: REGRESION LOGISTICA")
print(classification_report(y_valid, y_pred1, zero_division=0))
print(confusion_matrix(y_valid, y_pred1))

MODELO 1: REGRESION LOGISTICA
              precision    recall  f1-score   support

           0       0.97      0.88      0.92        40
           1       0.79      0.95      0.86        20

    accuracy                           0.90        60
   macro avg       0.88      0.91      0.89        60
weighted avg       0.91      0.90      0.90        60

[[35  5]
 [ 1 19]]


## Modelo 2: Árbol de Decisión
- Algoritmo que divide los datos mediante una secuencia de condiciones sobre las variables
- Cada división crea grupos más homogéneos (dentro de cada grupo, los valores de y se parecen más entre sí) hasta llegar a hojas que producen una predicción
- Sirve para clasificación y regresión
- Cuando utilizarlo: cuando interesan reglas comprensibles y relaciones no lineales
- Limitación: puede sobreajustar y cambiar bastante ante pequeñas variaciones en los datos
    - Sobreajustar: porque sigue creando divisiones hasta aprender detalles particulares del entrenamiento, regla muy específica si es muy profundo
- Escalado: no lo necesita

In [53]:
modelo2 = DecisionTreeClassifier(
    max_depth=3,     #Profundidad máxima del árbol, más profundidad-reglas más complejas, con riesgo de sobreajuste
    random_state=42,  #Exige al menos 5 observaciones en cada hoja
    min_samples_leaf=5, #Mínimo de observaciones en cada hoja/ grupo final
)

modelo2.fit(X_train_prep, y_train)

y_pred2 = modelo2.predict(X_valid_prep)

print("MODELO 2: ARBOL DE DECISION")
print(classification_report(y_valid, y_pred2, zero_division=0))
print(confusion_matrix(y_valid, y_pred2))

MODELO 2: ARBOL DE DECISION
              precision    recall  f1-score   support

           0       0.82      0.82      0.82        40
           1       0.65      0.65      0.65        20

    accuracy                           0.77        60
   macro avg       0.74      0.74      0.74        60
weighted avg       0.77      0.77      0.77        60

[[33  7]
 [ 7 13]]


## Modelo 3: Random Forest
- Algoritmo supervisado que combina muchos árboles de decisión
- Cada árbol se entrena con una muestras aleatoria de datos y considera un subconjunto aleatorio de variables en cada decisión
- Introduce 2 tipos de azar para que los árboles aprendan de formas diferentes
  1. Cada árbol recibe una muestra diferente de las filas de entrenamiento
  2. En cada división solo se selecciona al azar un grupo de variables
- Cuando utilizarlo:
    - Para capturar relaciones no lineales: cuando el efecto de una variable no sigue una pendiente cte
    - Interacciones: el efecto de una variable depende del valor de otra
- Limitación: es menos interpretable y consume más recursos que un árbol individual
- Escalado: no lo necesita

In [54]:
modelo3 = RandomForestClassifier(
    n_estimators=100,     #Número de árboles: más árboles estabilizan la predicción pero aumentan el coste de cálculo y acaba aportando poco
    max_depth=5,         #Profundidad máxima (decisiones consecutivas que toma un árbol antes de llegar a una hoja), menor profundidad limita la complejidad
    min_samples_leaf=5,  #Mínimo de observaciones por hoja
    max_features="sqrt", #Cuántas variables se consideran candidatas en cada división, consideras la raíz cuadrada del número de variables de entrada en cada división
    random_state=42
)

modelo3.fit(X_train_prep, y_train)

y_pred3 = modelo3.predict(X_valid_prep)

print("MODELO 3: RANDOM FOREST")
print(classification_report(y_valid, y_pred3, zero_division=0))
print(confusion_matrix(y_valid, y_pred3))

MODELO 3: RANDOM FOREST
              precision    recall  f1-score   support

           0       0.90      0.88      0.89        40
           1       0.76      0.80      0.78        20

    accuracy                           0.85        60
   macro avg       0.83      0.84      0.83        60
weighted avg       0.85      0.85      0.85        60

[[35  5]
 [ 4 16]]


## Modelo 4: KNN
- Algoritmo que predice a partir de las K observaciones más cercanas según una medida de distancia
- En clasificación utiliza sus clases, en regresión usa sus valores
- Los vecinos pueden contribuir por igual o ponderarse según su distancia
- Cuando utilizarlo: cuando observaciones cercanas tienden a tener resultados similares y el número de variables es moderado
- Limitación: es sensible a la escala, a variables irrelevantes y a la alta dimensionalidad. Predecir puede resultar muy costoso con muchos datos
- Escalado: importante si las variables tienen escalas diferentes

In [55]:
modelo4 = KNeighborsClassifier(
    n_neighbors=5,      #Número de vecinos utilizados, menos vecinos hacen la predicción más sensible al ruido, muchos suavizan la predicción
    weights="uniform",  #Cómo contribuye cada vecino. uniform: todos pesan igual, distance: los cercanos pesan más
    p=2)                #Tipo de distancia que se utiliza. 1: Manhattan, 2: euclídea

modelo4.fit(X_train_prep, y_train)

y_pred4 = modelo4.predict(X_valid_prep)

print("MODELO 4: KNN")
print(classification_report(y_valid, y_pred4, zero_division=0))
print(confusion_matrix(y_valid, y_pred4))

/lib/python3.13/site-packages/threadpoolctl.py:1932: UserWarning: Unable to import LDSO from pyodide_js._module. This should never happen.
  warnings.warn(


MODELO 4: KNN
              precision    recall  f1-score   support

           0       0.90      0.93      0.91        40
           1       0.84      0.80      0.82        20

    accuracy                           0.88        60
   macro avg       0.87      0.86      0.87        60
weighted avg       0.88      0.88      0.88        60

[[37  3]
 [ 4 16]]


## Elección de los modelos
El mejor modelo depende del coste de equivocarse:
- Si es caro dejar escapar positivos reales: prioriza recall, controlando las falsas alarmas.
- Si es caro actuar sobre falsas alarmas: prioriza precision, comprobando cuántos positivos dejas sin detectar.
- Si buscas equilibrar precision y recall: utiliza F1 como resumen.
- Si las clases están equilibradas y ambos errores tienen costes parecidos: accuracy puede ser útil.
- Si conoces el coste de cada tipo de error: compara el coste total esperado.
- Si dos modelos rinden parecido: considera su simplicidad, estabilidad, facilidad de explicación y tiempo de predicción.

En este caso nos preocupa no detectar los retrasos así que priorizo el recall, el F1 y el total de retrasos no detectados

### Ajustamos los hiperparámetros: entreno con train y eliges con valid

In [56]:
#Pruebo mayor regularización

modelo1_a = LogisticRegression(
    C=0.1,
    class_weight="balanced",
    max_iter=1000
)

modelo1_a.fit(X_train_prep, y_train)

y_pred1_a = modelo1_a.predict(X_valid_prep)

print("LOGISTICA: C=0.1")
print(classification_report(y_valid, y_pred1_a, zero_division=0))
print(confusion_matrix(y_valid, y_pred1_a))

LOGISTICA: C=0.1
              precision    recall  f1-score   support

           0       0.97      0.80      0.88        40
           1       0.70      0.95      0.81        20

    accuracy                           0.85        60
   macro avg       0.84      0.88      0.84        60
weighted avg       0.88      0.85      0.85        60

[[32  8]
 [ 1 19]]


In [57]:
#Pruebo con menor regularización

modelo1_b = LogisticRegression(
    C=10,
    class_weight="balanced",
    max_iter=1000
)

modelo1_b.fit(X_train_prep, y_train)

y_pred1_b = modelo1_b.predict(X_valid_prep)

print("LOGISTICA: C=10")
print(classification_report(y_valid, y_pred1_b, zero_division=0))
print(confusion_matrix(y_valid, y_pred1_b))

LOGISTICA: C=10
              precision    recall  f1-score   support

           0       0.97      0.85      0.91        40
           1       0.76      0.95      0.84        20

    accuracy                           0.88        60
   macro avg       0.87      0.90      0.88        60
weighted avg       0.90      0.88      0.89        60

[[34  6]
 [ 1 19]]


## Evaluación del modelo final

In [58]:
# Preparo test con el preprocesador que aprendió de train
X_test_prep = preprocesador.transform(X_test)

# Predigo con el modelo elegido
y_pred_test = modelo1.predict(X_test_prep)

# Evalúo
print(classification_report(y_test, y_pred_test, zero_division=0))
print(confusion_matrix(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93        40
           1       0.85      0.85      0.85        20

    accuracy                           0.90        60
   macro avg       0.89      0.89      0.89        60
weighted avg       0.90      0.90      0.90        60

[[37  3]
 [ 3 17]]


El modelo seleccionado es una regresión logística, elegida mediante la comparación de modelos y configuraciones en validación.

En los 60 pedidos de test obtiene un accuracy del 90 %. Detecta 17 de los 20 retrasos reales, lo que corresponde a un recall del 85 %. Genera tres falsas alarmas y deja tres retrasos sin detectar.

El modelo mejora la referencia de predecir siempre entregas puntuales, cuyo accuracy sería del 66,7 % y que no detectaría ningún retraso. Podría ayudar a priorizar la revisión de pedidos, aunque habría que confirmar si operaciones acepta el equilibrio entre retrasos no detectados y falsas alarmas.

La evaluación es limitada por el tamaño del dataset y el reparto aleatorio. Antes de utilizarlo con pedidos reales, comprobaría su rendimiento con datos posteriores en el tiempo y analizaría los casos en los que falla.

# ¿Qué pasaría si fuera un problema de Regresión?
- Aquí estimamos cantidades concretas
- Cambiamos la variable de target a la que quiera predecir
- En separación: quitamos stratify
- Modelos: utilizamos sus variables de regresión
- Evaluación: comparas MAE, RMSE y R2, ya no usamos precision, recall ni matriz de confusión

## Modelo: Regresión Lineal

In [60]:
modelo1 = LinearRegression()

modelo1.fit(X_train_prep, y_train)

y_pred1 = modelo1.predict(X_valid_prep)

print("MAE:", mean_absolute_error(y_valid, y_pred1))
print("RMSE:", np.sqrt(mean_squared_error(y_valid, y_pred1)))
print("R²:", r2_score(y_valid, y_pred1))

MAE: 0.2914879200049754
RMSE: 0.34002755672436374
R²: 0.47971567300626816


## Modelo: Árbol de Regresión

In [62]:
modelo2 = DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)

modelo2.fit(X_train_prep, y_train)

y_pred2 = modelo2.predict(X_valid_prep)

print("MAE:", mean_absolute_error(y_valid, y_pred2))
print("RMSE:", np.sqrt(mean_squared_error(y_valid, y_pred2)))
print("R²:", r2_score(y_valid, y_pred2))

MAE: 0.23983856758300023
RMSE: 0.37615260227605446
R²: 0.3632914891042859


## Modelo: Random Forest de Regresión

In [63]:
modelo3 = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

modelo3.fit(X_train_prep, y_train)

y_pred3 = modelo3.predict(X_valid_prep)

print("MAE:", mean_absolute_error(y_valid, y_pred3))
print("RMSE:", np.sqrt(mean_squared_error(y_valid, y_pred3)))
print("R²:", r2_score(y_valid, y_pred3))

MAE: 0.2000819460839686
RMSE: 0.3237892618793424
R²: 0.5282223125123378


## Modelo: KNN de Regresión

In [64]:
modelo4 = KNeighborsRegressor(
    n_neighbors=5,
    weights="uniform",
    p=2
)

modelo4.fit(X_train_prep, y_train)

y_pred4 = modelo4.predict(X_valid_prep)

print("MAE:", mean_absolute_error(y_valid, y_pred4))
print("RMSE:", np.sqrt(mean_squared_error(y_valid, y_pred4)))
print("R²:", r2_score(y_valid, y_pred4))

MAE: 0.2333333333333333
RMSE: 0.3306559138036599
R²: 0.5079999999999999
